# Anomaly Detection for Intrusion - Conceptual Exploration

## Introduction

The goal of this notebook is to explore methods for detecting anomalous human activity around vehicles (e.g., trucks), complementing the primary Faster-RCNN human detection. This involves identifying patterns or behaviors that deviate from an established norm, which could signify a potential intrusion or security concern.

## Defining Anomalies

Anomalies in human activity can be categorized in several ways:

- **Spatial Anomalies:** Relate to the location of humans.
  - Humans entering predefined restricted zones (e.g., too close to the truck's cargo area, undercarriage, fuel tank at certain times).
  - Humans detected in unusual proximity to specific truck parts (e.g., lingering near wheels or engine compartment for extended periods).
  - Humans appearing on an unexpected side of the truck or at an unusual entry/exit point relative to a facility.

- **Temporal Anomalies:** Relate to the timing and duration of activities.
  - Loitering: Humans remaining in an area for an unusually long time, especially if not actively engaged in a recognized task.
  - Sudden Appearance/Disappearance: Humans appearing or vanishing in a way that bypasses expected entry/exit points (might require robust tracking).
  - Unusual Speed or Movement Patterns: Humans moving too quickly, too slowly, or in erratic patterns (e.g., pacing).
  - Activity at Odd Hours: Presence detected during times when no activity is expected (e.g., late at night in a secured yard).

- **Object-based Anomalies (Optional/Advanced):**
  - Unusual Pose/Stature: While harder to define and detect reliably, sustained unusual postures (e.g., crouching for a long time) could be flagged. This often overlaps with action recognition.
  - Presence of Unexpected Objects: Humans carrying unusual tools or objects (this would require object detection beyond just humans).

## Feature Engineering Ideas

To detect anomalies, we first need to extract relevant features from the human detection data:

- **From Bounding Boxes (per detected human):
  - Centroid (x, y coordinates).
  - Size (width, height, area).
  - Aspect Ratio (width/height).
  - Velocity and Acceleration (if tracking is implemented across frames).
  - Direction of movement (if tracking is implemented).

- **Relationship to Regions of Interest (ROIs):
  - Whether a human's bounding box (or centroid) is inside, outside, or overlapping with a user-defined ROI (e.g., a forbidden zone drawn in the Streamlit app).
  - Distance to the nearest ROI boundary.

- **Aggregate Features (per frame or time window):
  - Count of detected people.
  - Density of people in a specific area.
  - Change in the count of people.

- **Frame-level Features (especially if no humans are detected but suspicion remains - Advanced):
  - Optical flow magnitude/direction within an ROI (e.g., to detect motion near a truck even if the human isn't clearly segmented).
  - Background subtraction changes within an ROI.

In [ ]:
# PSEUDO-CODE: Extracting bounding box centroids

# Assume detected_humans is a list of dictionaries, where each dict contains:
# 'box': [x1, y1, x2, y2] coordinates of the bounding box
# 'frame': frame number in which the human was detected
# 'label': class label (e.g., 'person')
# 'score': confidence score of the detection

# Example: detected_humans = [{'box': [10, 20, 50, 120], 'frame': 1, 'label': 'person', 'score': 0.9}, ...]

def extract_centroid_features(detected_humans_data):
    features = []
    for detection in detected_humans_data:
        box = detection['box']
        frame_num = detection['frame']
        
        centroid_x = (box[0] + box[2]) / 2
        centroid_y = (box[1] + box[3]) / 2
        
        features.append({
            'frame': frame_num,
            'centroid_x': centroid_x,
            'centroid_y': centroid_y,
            'original_box': box,
            'score': detection.get('score') # Include score if available
        })
    return features

# Example usage:
# detected_humans_sample = [
#     {'box': [10, 20, 50, 120], 'frame': 1, 'score': 0.95},
#     {'box': [15, 25, 55, 125], 'frame': 2, 'score': 0.92}
# ]
# centroid_features = extract_centroid_features(detected_humans_sample)
# print(centroid_features)

## Potential Anomaly Detection Algorithms

Once features are extracted, various algorithms can be used to identify anomalies. Some candidates include:

- **Isolation Forest:**
  - Particularly well-suited for this type of problem.
  - Works on the principle that anomalies are "few and different" and thus easier to isolate in a tree structure.
  - Generally handles high-dimensional data well and is computationally efficient.
  - Does not require assumptions about data distribution (non-parametric).
  - Good starting point for many anomaly detection tasks.

- **One-Class SVM:**
  - Learns a boundary that encompasses the "normal" data points.
  - Points falling outside this boundary are considered anomalies.
  - Can be effective but might be sensitive to parameter tuning (e.g., kernel, nu).

- **Local Outlier Factor (LOF):**
  - Measures the local density deviation of a data point with respect to its neighbors.
  - Anomalies are points in lower-density regions.
  - Effective for detecting anomalies that are outliers relative to their local neighborhood.
  - Can be computationally more expensive for large datasets.

- **Gaussian Mixture Models (GMMs):**
  - Assumes data is generated from a mixture of Gaussian distributions.
  - Can model complex distributions of normal data.
  - Anomalies are points that have a low probability under the fitted GMM.
  - Requires estimating the number of components (Gaussians).

- **Autoencoders (Deep Learning Approach - more complex):
  - Neural networks trained to reconstruct normal data.
  - Anomalies are data points that the autoencoder struggles to reconstruct accurately (resulting in a high reconstruction error).
  - Powerful for complex patterns but requires more data and computational resources.

## Initial Experimentation Strategy (Conceptual)

1.  **Data Acquisition/Simulation:**
    - Use the output from `scripts.process_video.py` (which provides bounding boxes per frame from human detection) as a source of real data.
    - Alternatively, for initial tests, manually create sample bounding box data (lists of `{'box': [x,y,w,h], 'frame': i, ...}` dictionaries) representing various scenarios.
    - The Streamlit app's ROI definition can provide coordinates for forbidden zones.

2.  **Define a Simple Scenario:**
    - **Scenario:** Detecting human entry into a static, predefined "forbidden zone" ROI.
    - **"Normal" Data:** Human detections (centroids) that are outside this forbidden zone.
    - **"Anomalous" Data:** Human detections (centroids) that fall inside this forbidden zone.

3.  **Feature Extraction for this Scenario:**
    - For each detected human, extract the centroid (x, y).
    - (Optional) Add a binary feature: `is_in_forbidden_zone`.

4.  **Model Application (e.g., Isolation Forest):
    - Train an Isolation Forest model on features from predominantly "normal" scenarios (or a mix, as IF doesn't strictly need pure normal data for training if anomalies are rare).
    - For a new detection, predict its anomaly score. Points inside the forbidden zone should ideally receive higher anomaly scores.

5.  **Evaluation (Conceptual):
    - Manually label some test data (frames with/without intrusion into the zone).
    - See if the anomaly scores from the model correlate with these labels.
    - Adjust model parameters (e.g., `contamination` in Isolation Forest) as needed.

In [ ]:
# PSEUDO-CODE: Checking if a centroid is within a static ROI

def check_spatial_anomaly(centroid_features, forbidden_roi):
    """
    Checks if human centroids fall within a defined forbidden ROI.
    Adds an 'is_anomalous_spatial' flag to each feature point.
    
    Args:
        centroid_features: List of dicts, e.g., [{'frame': ..., 'centroid_x': ..., 'centroid_y': ...}]
        forbidden_roi: Dict, e.g., {'x_min': 100, 'y_min': 100, 'x_max': 200, 'y_max': 200}
    """
    updated_features = []
    for feature_point in centroid_features:
        cx = feature_point['centroid_x']
        cy = feature_point['centroid_y']
        
        is_in_roi = (forbidden_roi['x_min'] <= cx <= forbidden_roi['x_max'] and
                       forbidden_roi['y_min'] <= cy <= forbidden_roi['y_max'])
        
        feature_point['is_anomalous_spatial'] = is_in_roi
        updated_features.append(feature_point)
    return updated_features

# Example usage:
# sample_features = [
#     {'frame': 1, 'centroid_x': 50, 'centroid_y': 50},
#     {'frame': 1, 'centroid_x': 150, 'centroid_y': 150}, # Inside ROI
#     {'frame': 2, 'centroid_x': 250, 'centroid_y': 250}
# ]
# defined_roi = {'x_min': 100, 'y_min': 100, 'x_max': 200, 'y_max': 200}

# features_with_anomaly_flag = check_spatial_anomaly(sample_features, defined_roi)
# for f in features_with_anomaly_flag:
#     if f['is_anomalous_spatial']:
#         print(f"Frame {f['frame']}: Spatial anomaly detected at ({f['centroid_x']}, {f['centroid_y']})")

## Next Steps (Conceptual)

The immediate practical next steps would involve:

1.  **Implement Data Loading:** Write Python code to load bounding box data from actual video processing outputs or simulated CSV/JSON files.
2.  **Feature Extraction:** Implement the centroid calculation and potentially other chosen features (e.g., size, relation to user-drawn ROI from Streamlit app).
3.  **Basic Anomaly Model:** Implement the spatial anomaly check (centroid in static ROI) as a baseline.
4.  **Isolation Forest Implementation:** 
    - Train an Isolation Forest model on a dataset of these features.
    - Experiment with the `contamination` parameter.
    - Evaluate its ability to flag the simple spatial anomalies and potentially more subtle ones if more complex features are used.
5.  **Visualization:** Plot feature distributions and highlight detected anomalies in scatter plots or on video frames.
6.  **Integration Strategy:** Think about how an anomaly score or flag could be passed back to the Streamlit application for display (e.g., highlighting an anomalous bounding box or showing an alert).